# v0.5 Ensemble (Rung 4)

See `docs/plans/v0.5-ensemble-plan.md`. Current best: v0.3 CatBoost (either variant,
statistically tied) OOF ~0.949 / LB ~0.949.

Per `implementation-plan.md`'s Rung 4: blend LightGBM + CatBoost + a regularized
linear baseline. Given the Rung 3 finding that this dataset likely has a ~0.95
synthesis-noise ceiling (see `docs/investigate/2026-07-03-kaggle-discussion-findings.md`),
and that LightGBM/CatBoost already converge to a similar range (suggesting
correlated errors), the realistic upside here is probably small — worth trying
since it's cheap, but reported honestly either way.

**Four members**, each reproducing an exact validated config (no re-tuning — Rung 4
is about combining existing best efforts) capturing OOF *and* test probabilities
this time, since v0.1/v0.3's original harnesses only saved hard labels for OOF:
1. **LightGBM** — v0.1's exact config, base features.
2. **CatBoost Variant 1** — v0.3's exact config, base features.
3. **CatBoost Variant 2** — v0.3's exact config, v0.2's 35-feature engineered set —
   same algorithm as #2 but a different feature view, which may still add some
   diversity even though the two variants are statistically tied in aggregate score.
4. **Regularized logistic regression** (new) — genuine architectural diversity from
   the tree models.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

DATA_DIR = Path('..') / 'data'
TARGET = 'health_condition'
NUMERIC_COLS = ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure',
                 'step_count', 'exercise_duration', 'water_intake']
CATEGORICAL_COLS = ['diet_type', 'stress_level', 'sleep_quality',
                     'physical_activity_level', 'smoking_alcohol', 'gender']
BASE_FEATURES = NUMERIC_COLS + CATEGORICAL_COLS

train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')

for col in CATEGORICAL_COLS:
    train[col] = train[col].fillna('missing').astype(str)
    test[col] = test[col].fillna('missing').astype(str)

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(train[TARGET])
n_classes = len(label_encoder.classes_)
print('classes:', list(label_encoder.classes_))

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
folds = list(skf.split(train, y))
print('train', train.shape, 'test', test.shape)

sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

classes: ['at-risk', 'fit', 'unhealthy']
train (690088, 15) test (295753, 14)


## Member 1 — LightGBM (reproducing v0.1's exact config)

LightGBM needs `pandas.Categorical` dtype with a shared category list between train
and test (unlike CatBoost, which hashes raw strings) — build a separate categorical
encoding just for this model.

In [2]:
train_lgb = train.copy()
test_lgb = test.copy()
for col in CATEGORICAL_COLS:
    categories = sorted(set(train_lgb[col].unique()) | set(test_lgb[col].unique()))
    train_lgb[col] = pd.Categorical(train_lgb[col], categories=categories)
    test_lgb[col] = pd.Categorical(test_lgb[col], categories=categories)

class TqdmLgbCallback:
    def __init__(self, total, desc):
        self.pbar = tqdm(total=total, desc=desc, leave=False)
    def __call__(self, env):
        self.pbar.update(1)

oof_proba_lgb = np.zeros((len(train_lgb), n_classes))
test_proba_lgb = np.zeros((len(test_lgb), n_classes))
lgb_fold_scores, lgb_best_iters = [], []

for fold, (tr_idx, val_idx) in enumerate(tqdm(folds, desc='LightGBM folds')):
    X_tr, X_val = train_lgb[BASE_FEATURES].iloc[tr_idx], train_lgb[BASE_FEATURES].iloc[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]
    model = lgb.LGBMClassifier(
        objective='multiclass', num_class=n_classes, class_weight='balanced',
        n_estimators=2000, learning_rate=0.05, num_leaves=63,
        subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1, verbosity=-1,
    )
    round_pbar = TqdmLgbCallback(2000, f'fold {fold} rounds')
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], eval_metric='multi_logloss',
              categorical_feature=CATEGORICAL_COLS,
              callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False), lgb.log_evaluation(0), round_pbar])
    round_pbar.pbar.close()
    oof_proba_lgb[val_idx] = model.predict_proba(X_val, num_iteration=model.best_iteration_)
    val_pred = oof_proba_lgb[val_idx].argmax(axis=1)
    lgb_fold_scores.append(balanced_accuracy_score(y_val, val_pred))
    lgb_best_iters.append(model.best_iteration_)
    test_proba_lgb += model.predict_proba(test_lgb[BASE_FEATURES], num_iteration=model.best_iteration_) / N_FOLDS

oof_pred_lgb = oof_proba_lgb.argmax(axis=1)
oof_bal_acc_lgb = balanced_accuracy_score(y, oof_pred_lgb)
print('best_iterations:', lgb_best_iters)
print('fold scores:', [round(s, 4) for s in lgb_fold_scores])
print(f'LightGBM OOF balanced accuracy: {oof_bal_acc_lgb:.4f}')
print('v0.1 known result:             0.9389')

LightGBM folds:   0%|          | 0/5 [00:00<?, ?it/s]

fold 0 rounds:   0%|          | 0/2000 [00:00<?, ?it/s]

fold 1 rounds:   0%|          | 0/2000 [00:00<?, ?it/s]

fold 2 rounds:   0%|          | 0/2000 [00:00<?, ?it/s]

fold 3 rounds:   0%|          | 0/2000 [00:00<?, ?it/s]

fold 4 rounds:   0%|          | 0/2000 [00:00<?, ?it/s]

best_iterations: [2000, 2000, 2000, 2000, 2000]
fold scores: [0.9397, 0.9402, 0.9389, 0.9389, 0.9367]
LightGBM OOF balanced accuracy: 0.9389
v0.1 known result:             0.9389


In [3]:
assert abs(oof_bal_acc_lgb - 0.9389) < 0.002, 'LightGBM reproduction did not match v0.1 closely enough'
print('LightGBM reproduction check: PASS')

LightGBM reproduction check: PASS


## CatBoost shared setup (custom eval metric + progress callback)

Used by both CatBoost members (Variant 1 and Variant 2).

In [4]:
class BalancedAccuracyMetric:
    def is_max_optimal(self):
        return True
    def evaluate(self, approxes, target, weight):
        approx = np.array(approxes)
        pred = approx.argmax(axis=0)
        y_true = np.array(target).astype(int)
        return balanced_accuracy_score(y_true, pred), 1.0
    def get_final_error(self, error, weight):
        return error

class TqdmCatBoostCallback:
    def __init__(self, total, desc):
        self.pbar = tqdm(total=total, desc=desc, leave=False)
        self.last = 0
    def after_iteration(self, info):
        self.pbar.update(info.iteration - self.last)
        self.last = info.iteration
        return True

def run_catboost_cv_with_proba(feature_frame, test_frame, categorical_features, folds, y, n_classes, desc):
    oof_proba = np.zeros((len(feature_frame), n_classes))
    test_proba = np.zeros((len(test_frame), n_classes))
    fold_scores, best_iters = [], []
    for fold, (tr_idx, val_idx) in enumerate(tqdm(folds, desc=desc)):
        X_tr, X_val = feature_frame.iloc[tr_idx], feature_frame.iloc[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]
        model = CatBoostClassifier(
            loss_function='MultiClass', eval_metric=BalancedAccuracyMetric(),
            auto_class_weights='Balanced', iterations=3000, learning_rate=0.05, depth=6,
            l2_leaf_reg=3, random_seed=42, task_type='CPU', verbose=False,
        )
        round_pbar = TqdmCatBoostCallback(3000, f'{desc} fold {fold} rounds')
        model.fit(X_tr, y_tr, cat_features=categorical_features, eval_set=(X_val, y_val),
                  early_stopping_rounds=100, callbacks=[round_pbar])
        round_pbar.pbar.close()
        oof_proba[val_idx] = model.predict_proba(X_val)
        val_pred = oof_proba[val_idx].argmax(axis=1)
        fold_scores.append(balanced_accuracy_score(y_val, val_pred))
        best_iters.append(model.best_iteration_)
        test_proba += model.predict_proba(test_frame) / len(folds)
    oof_pred = oof_proba.argmax(axis=1)
    oof_bal_acc = balanced_accuracy_score(y, oof_pred)
    return {'oof_proba': oof_proba, 'test_proba': test_proba, 'fold_scores': fold_scores,
            'best_iters': best_iters, 'oof_balanced_accuracy': oof_bal_acc}

## Member 2 — CatBoost Variant 1 (reproducing v0.3's exact config, base features)

In [5]:
result_cb_v1 = run_catboost_cv_with_proba(train[BASE_FEATURES], test[BASE_FEATURES], CATEGORICAL_COLS,
                                           folds, y, n_classes, desc='CatBoost-V1')
print('best_iterations:', result_cb_v1['best_iters'])
print('fold scores:', [round(s, 4) for s in result_cb_v1['fold_scores']])
print(f"CatBoost-V1 OOF balanced accuracy: {result_cb_v1['oof_balanced_accuracy']:.4f}")
print('v0.3 Variant 1 known result:       0.9493, best_iterations [428, 950, 605, 339, 779]')

CatBoost-V1:   0%|          | 0/5 [00:00<?, ?it/s]

CatBoost-V1 fold 0 rounds:   0%|          | 0/3000 [00:00<?, ?it/s]

/Users/marksusol/LosusAI/Projects/Kaggle/.venv/lib/python3.12/site-packages/catboost/core.py:1824: UserWarning: Failed to optimize method "evaluate" in the passed object:
Failed in nopython mode pipeline (step: nopython frontend)
Untyped global name 'balanced_accuracy_score': Cannot determine Numba type of <class 'function'>

File "../../../../../../../var/folders/c3/gnpb2cfd5znckvplsjxhcy2r0000gn/T/ipykernel_44119/2562798231.py", line 8:
<source missing, REPL/exec in use?>

During: Pass nopython_type_inference
  self._object._train(train_pool, test_pool, params, allow_clear_pool, init_model._object if init_model else None)


CatBoost-V1 fold 1 rounds:   0%|          | 0/3000 [00:00<?, ?it/s]

/Users/marksusol/LosusAI/Projects/Kaggle/.venv/lib/python3.12/site-packages/catboost/core.py:1824: UserWarning: Failed to optimize method "evaluate" in the passed object:
Failed in nopython mode pipeline (step: nopython frontend)
Untyped global name 'balanced_accuracy_score': Cannot determine Numba type of <class 'function'>

File "../../../../../../../var/folders/c3/gnpb2cfd5znckvplsjxhcy2r0000gn/T/ipykernel_44119/2562798231.py", line 8:
<source missing, REPL/exec in use?>

During: Pass nopython_type_inference
  self._object._train(train_pool, test_pool, params, allow_clear_pool, init_model._object if init_model else None)


...

CatBoost-V1 fold 3 rounds:   0%|          | 0/3000 [00:00<?, ?it/s]

/Users/marksusol/LosusAI/Projects/Kaggle/.venv/lib/python3.12/site-packages/catboost/core.py:1824: UserWarning: Failed to optimize method "evaluate" in the passed object:
Failed in nopython mode pipeline (step: nopython frontend)
Untyped global name 'balanced_accuracy_score': Cannot determine Numba type of <class 'function'>

File "../../../../../../../var/folders/c3/gnpb2cfd5znckvplsjxhcy2r0000gn/T/ipykernel_44119/2562798231.py", line 8:
<source missing, REPL/exec in use?>

During: Pass nopython_type_inference
  self._object._train(train_pool, test_pool, params, allow_clear_pool, init_model._object if init_model else None)


CatBoost-V1 fold 4 rounds:   0%|          | 0/3000 [00:00<?, ?it/s]

/Users/marksusol/LosusAI/Projects/Kaggle/.venv/lib/python3.12/site-packages/catboost/core.py:1824: UserWarning: Failed to optimize method "evaluate" in the passed object:
Failed in nopython mode pipeline (step: nopython frontend)
Untyped global name 'balanced_accuracy_score': Cannot determine Numba type of <class 'function'>

File "../../../../../../../var/folders/c3/gnpb2cfd5znckvplsjxhcy2r0000gn/T/ipykernel_44119/2562798231.py", line 8:
<source missing, REPL/exec in use?>

During: Pass nopython_type_inference
  self._object._train(train_pool, test_pool, params, allow_clear_pool, init_model._object if init_model else None)


best_iterations: [428, 950, 605, 339, 779]
fold scores: [0.9497, 0.9513, 0.9487, 0.949, 0.9479]
CatBoost-V1 OOF balanced accuracy: 0.9493
v0.3 Variant 1 known result:       0.9493, best_iterations [428, 950, 605, 339, 779]


In [6]:
assert abs(result_cb_v1['oof_balanced_accuracy'] - 0.9493) < 0.002, 'CatBoost-V1 reproduction did not match v0.3 Variant 1 closely enough'
print('CatBoost-V1 reproduction check: PASS')

CatBoost-V1 reproduction check: PASS


## Member 3 — CatBoost Variant 2 (reproducing v0.3's exact config, v0.2's engineered features)

Reconstruct v0.2/v0.3 Variant 2's 35-feature engineered set (missingness indicators,
categorical crosses, OOF smoothed multiclass target encoding), same as v0.4.

In [7]:
train['sleep_duration_bin'], sleep_bin_edges = pd.qcut(
    train['sleep_duration'], q=10, duplicates='drop', retbins=True)
test_sleep_clipped = test['sleep_duration'].clip(lower=sleep_bin_edges[0], upper=sleep_bin_edges[-1])
test['sleep_duration_bin'] = pd.cut(test_sleep_clipped, bins=sleep_bin_edges, include_lowest=True)

train['sleep_duration_bin'] = train['sleep_duration_bin'].astype(str)
train.loc[train['sleep_duration'].isnull(), 'sleep_duration_bin'] = 'missing'
test['sleep_duration_bin'] = test['sleep_duration_bin'].astype(str)
test.loc[test['sleep_duration'].isnull(), 'sleep_duration_bin'] = 'missing'

MISSING_INDICATOR_COLS = []
for col in NUMERIC_COLS:
    ind_col = f'is_missing_{col}'
    train[ind_col] = train[col].isnull().astype(int)
    test[ind_col] = test[col].isnull().astype(int)
    MISSING_INDICATOR_COLS.append(ind_col)

def make_cross(df, col_a, col_b, name):
    df[name] = df[col_a].astype(str) + '_' + df[col_b].astype(str)

make_cross(train, 'stress_level', 'physical_activity_level', 'stress_x_activity')
make_cross(test, 'stress_level', 'physical_activity_level', 'stress_x_activity')
make_cross(train, 'sleep_quality', 'smoking_alcohol', 'sleepq_x_smoking')
make_cross(test, 'sleep_quality', 'smoking_alcohol', 'sleepq_x_smoking')
make_cross(train, 'sleep_duration_bin', 'stress_level', 'sleepbin_x_stress')
make_cross(test, 'sleep_duration_bin', 'stress_level', 'sleepbin_x_stress')

CROSS_COLS = ['stress_x_activity', 'sleepq_x_smoking', 'sleepbin_x_stress']

def oof_target_encode(train_col, test_col, y, n_classes, folds, alpha=20):
    n_train = len(train_col)
    oof_encoded = np.zeros((n_train, n_classes))
    test_encoded = np.zeros((len(test_col), n_classes))
    class_cols = [f'k{i}' for i in range(n_classes)]
    onehot = pd.get_dummies(pd.Series(y)).values
    for tr_idx, val_idx in folds:
        cats_tr = pd.Series(train_col.iloc[tr_idx].astype(str).values)
        y_tr_onehot = onehot[tr_idx]
        prior = pd.Series(y_tr_onehot.mean(axis=0), index=class_cols)
        stats = pd.DataFrame(y_tr_onehot, columns=class_cols)
        stats['cat'] = cats_tr.values
        grp_sum = stats.groupby('cat')[class_cols].sum()
        grp_count = stats.groupby('cat').size()
        enc_map = grp_sum.add(alpha * prior, axis=1).div(grp_count + alpha, axis=0)
        val_cats = pd.Series(train_col.iloc[val_idx].astype(str).values)
        oof_encoded[val_idx] = enc_map.reindex(val_cats).fillna(prior).values
        test_cats = pd.Series(test_col.astype(str).values)
        test_encoded += enc_map.reindex(test_cats).fillna(prior).values / len(folds)
    return oof_encoded, test_encoded

TARGET_ENCODE_COLS = ['stress_level', 'physical_activity_level', 'stress_x_activity', 'sleepq_x_smoking']
TARGET_ENCODED_FEATURE_COLS = []
for col in tqdm(TARGET_ENCODE_COLS, desc='target encoding'):
    oof_enc, test_enc = oof_target_encode(train[col], test[col], y, n_classes, folds)
    for k in range(n_classes):
        fcol = f'te_{col}_k{k}'
        train[fcol] = oof_enc[:, k]
        test[fcol] = test_enc[:, k]
        TARGET_ENCODED_FEATURE_COLS.append(fcol)

ENGINEERED_CATEGORICAL_COLS = CATEGORICAL_COLS + CROSS_COLS
ENGINEERED_FEATURES = (NUMERIC_COLS + CATEGORICAL_COLS + MISSING_INDICATOR_COLS
                        + CROSS_COLS + TARGET_ENCODED_FEATURE_COLS)
print(f'{len(ENGINEERED_FEATURES)} engineered features (matches v0.2/v0.3 Variant 2)')

target encoding:   0%|          | 0/4 [00:00<?, ?it/s]

35 engineered features (matches v0.2/v0.3 Variant 2)


In [8]:
result_cb_v2 = run_catboost_cv_with_proba(train[ENGINEERED_FEATURES], test[ENGINEERED_FEATURES],
                                           ENGINEERED_CATEGORICAL_COLS, folds, y, n_classes, desc='CatBoost-V2')
print('best_iterations:', result_cb_v2['best_iters'])
print('fold scores:', [round(s, 4) for s in result_cb_v2['fold_scores']])
print(f"CatBoost-V2 OOF balanced accuracy: {result_cb_v2['oof_balanced_accuracy']:.4f}")
print('v0.3 Variant 2 known result:       0.9491, best_iterations [544, 765, 542, 354, 628]')

CatBoost-V2:   0%|          | 0/5 [00:00<?, ?it/s]

CatBoost-V2 fold 0 rounds:   0%|          | 0/3000 [00:00<?, ?it/s]

/Users/marksusol/LosusAI/Projects/Kaggle/.venv/lib/python3.12/site-packages/catboost/core.py:1824: UserWarning: Failed to optimize method "evaluate" in the passed object:
Failed in nopython mode pipeline (step: nopython frontend)
Untyped global name 'balanced_accuracy_score': Cannot determine Numba type of <class 'function'>

File "../../../../../../../var/folders/c3/gnpb2cfd5znckvplsjxhcy2r0000gn/T/ipykernel_44119/2562798231.py", line 8:
<source missing, REPL/exec in use?>

During: Pass nopython_type_inference
  self._object._train(train_pool, test_pool, params, allow_clear_pool, init_model._object if init_model else None)


CatBoost-V2 fold 1 rounds:   0%|          | 0/3000 [00:00<?, ?it/s]

/Users/marksusol/LosusAI/Projects/Kaggle/.venv/lib/python3.12/site-packages/catboost/core.py:1824: UserWarning: Failed to optimize method "evaluate" in the passed object:
Failed in nopython mode pipeline (step: nopython frontend)
Untyped global name 'balanced_accuracy_score': Cannot determine Numba type of <class 'function'>

File "../../../../../../../var/folders/c3/gnpb2cfd5znckvplsjxhcy2r0000gn/T/ipykernel_44119/2562798231.py", line 8:
<source missing, REPL/exec in use?>

During: Pass nopython_type_inference
  self._object._train(train_pool, test_pool, params, allow_clear_pool, init_model._object if init_model else None)


...

CatBoost-V2 fold 3 rounds:   0%|          | 0/3000 [00:00<?, ?it/s]

/Users/marksusol/LosusAI/Projects/Kaggle/.venv/lib/python3.12/site-packages/catboost/core.py:1824: UserWarning: Failed to optimize method "evaluate" in the passed object:
Failed in nopython mode pipeline (step: nopython frontend)
Untyped global name 'balanced_accuracy_score': Cannot determine Numba type of <class 'function'>

File "../../../../../../../var/folders/c3/gnpb2cfd5znckvplsjxhcy2r0000gn/T/ipykernel_44119/2562798231.py", line 8:
<source missing, REPL/exec in use?>

During: Pass nopython_type_inference
  self._object._train(train_pool, test_pool, params, allow_clear_pool, init_model._object if init_model else None)


CatBoost-V2 fold 4 rounds:   0%|          | 0/3000 [00:00<?, ?it/s]

/Users/marksusol/LosusAI/Projects/Kaggle/.venv/lib/python3.12/site-packages/catboost/core.py:1824: UserWarning: Failed to optimize method "evaluate" in the passed object:
Failed in nopython mode pipeline (step: nopython frontend)
Untyped global name 'balanced_accuracy_score': Cannot determine Numba type of <class 'function'>

File "../../../../../../../var/folders/c3/gnpb2cfd5znckvplsjxhcy2r0000gn/T/ipykernel_44119/2562798231.py", line 8:
<source missing, REPL/exec in use?>

During: Pass nopython_type_inference
  self._object._train(train_pool, test_pool, params, allow_clear_pool, init_model._object if init_model else None)


best_iterations: [544, 765, 542, 354, 628]
fold scores: [0.9498, 0.9511, 0.9482, 0.9486, 0.9481]
CatBoost-V2 OOF balanced accuracy: 0.9491
v0.3 Variant 2 known result:       0.9491, best_iterations [544, 765, 542, 354, 628]


In [9]:
assert abs(result_cb_v2['oof_balanced_accuracy'] - 0.9491) < 0.002, 'CatBoost-V2 reproduction did not match v0.3 Variant 2 closely enough'
print('CatBoost-V2 reproduction check: PASS')

CatBoost-V2 reproduction check: PASS


## Member 4 — Regularized logistic regression (new)

Genuine architectural diversity from the tree ensembles. Needs different
preprocessing: one-hot categoricals, median-impute numerics (linear models can't
handle `NaN` natively, unlike the tree models), standardize numeric features. Uses
base features (same as LightGBM/CatBoost-V1) for a clean comparison.

In [10]:
preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler()),
    ]), NUMERIC_COLS),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL_COLS),
])

oof_proba_lr = np.zeros((len(train), n_classes))
test_proba_lr = np.zeros((len(test), n_classes))
lr_fold_scores = []

for fold, (tr_idx, val_idx) in enumerate(tqdm(folds, desc='LogisticRegression folds')):
    X_tr, X_val = train[BASE_FEATURES].iloc[tr_idx], train[BASE_FEATURES].iloc[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]
    pipe = Pipeline([
        ('preprocess', preprocessor),
        ('clf', LogisticRegression(class_weight='balanced', C=1.0, max_iter=2000, random_state=42)),
    ])
    pipe.fit(X_tr, y_tr)
    oof_proba_lr[val_idx] = pipe.predict_proba(X_val)
    val_pred = oof_proba_lr[val_idx].argmax(axis=1)
    lr_fold_scores.append(balanced_accuracy_score(y_val, val_pred))
    test_proba_lr += pipe.predict_proba(test[BASE_FEATURES]) / N_FOLDS

oof_pred_lr = oof_proba_lr.argmax(axis=1)
oof_bal_acc_lr = balanced_accuracy_score(y, oof_pred_lr)
print('fold scores:', [round(s, 4) for s in lr_fold_scores])
print(f'LogisticRegression OOF balanced accuracy: {oof_bal_acc_lr:.4f}')
print()
print(classification_report(y, oof_pred_lr, target_names=label_encoder.classes_, digits=4))

LogisticRegression folds:   0%|          | 0/5 [00:00<?, ?it/s]

fold scores: [0.8999, 0.9002, 0.8996, 0.8991, 0.8982]
LogisticRegression OOF balanced accuracy: 0.8994

              precision    recall  f1-score   support

     at-risk     0.9893    0.8164    0.8946    592561
         fit     0.4439    0.9518    0.6055     39803
   unhealthy     0.4639    0.9300    0.6190     57724

    accuracy                         0.8337    690088
   macro avg     0.6324    0.8994    0.7063    690088
weighted avg     0.9139    0.8337    0.8548    690088



## Solo model comparison

In [11]:
OOF_PROBAS = {
    'lgbm': oof_proba_lgb,
    'catboost_v1': result_cb_v1['oof_proba'],
    'catboost_v2': result_cb_v2['oof_proba'],
    'logreg': oof_proba_lr,
}
TEST_PROBAS = {
    'lgbm': test_proba_lgb,
    'catboost_v1': result_cb_v1['test_proba'],
    'catboost_v2': result_cb_v2['test_proba'],
    'logreg': test_proba_lr,
}
MEMBER_NAMES = ['lgbm', 'catboost_v1', 'catboost_v2', 'logreg']

solo_scores = {
    'lgbm': oof_bal_acc_lgb,
    'catboost_v1': result_cb_v1['oof_balanced_accuracy'],
    'catboost_v2': result_cb_v2['oof_balanced_accuracy'],
    'logreg': oof_bal_acc_lr,
}
for name, score in sorted(solo_scores.items(), key=lambda kv: -kv[1]):
    print(f'{score:.4f}  {name}')

0.9493  catboost_v1
0.9491  catboost_v2
0.9389  lgbm
0.8994  logreg


## Blend weight search (4-way simplex grid) + nested validation

Blend = weighted average of the 4 OOF probability matrices, argmax. Weights don't
need to sum to 1 for argmax purposes (only ratios matter), but a simplex
parameterization (weights summing to 1) is a convenient way to grid-search the
ratio space. Using a coarser step (0.1) than Rung 3's pairwise search (0.02) to keep
the 4-dimensional grid's combinatorial size manageable. Nested validation (fit
weights on 4/5 folds, evaluate on the held-out 5th) gives the honest improvement
estimate, same discipline as Rung 3.

In [12]:
def simplex_grid(n_members, step=0.1):
    n_steps = round(1 / step)
    def rec(remaining_steps, remaining_members):
        if remaining_members == 1:
            yield (remaining_steps * step,)
            return
        for i in range(remaining_steps + 1):
            for rest in rec(remaining_steps - i, remaining_members - 1):
                yield (i * step,) + rest
    return list(rec(n_steps, n_members))

def blend_predict(probas_dict, weights, members):
    blended = sum(w * probas_dict[m] for w, m in zip(weights, members))
    return blended.argmax(axis=1)

def grid_search_blend(oof_probas_dict, y_true, members, grid):
    best_score, best_w = -1, tuple(1 / len(members) for _ in members)
    for w in tqdm(grid, desc=f'blend grid ({"+".join(members)})'):
        pred = blend_predict(oof_probas_dict, w, members)
        score = balanced_accuracy_score(y_true, pred)
        if score > best_score:
            best_score, best_w = score, w
    return best_score, best_w

grid4 = simplex_grid(len(MEMBER_NAMES), step=0.1)
print(f'4-way grid size: {len(grid4)} combinations')
best_score_4way, best_w_4way = grid_search_blend(OOF_PROBAS, y, MEMBER_NAMES, grid4)
print(f'4-way blend full-OOF grid search: best {best_score_4way:.4f} at weights {dict(zip(MEMBER_NAMES, (round(x,2) for x in best_w_4way)))}')
print(f'Best solo model:                  {max(solo_scores.values()):.4f}')
print(f'Improvement (same-data, likely optimistic): {best_score_4way - max(solo_scores.values()):+.4f}')

4-way grid size: 286 combinations


blend grid (lgbm+catboost_v1+catboost_v2+logreg):   0%|          | 0/286 [00:00<?, ?it/s]

4-way blend full-OOF grid search: best 0.9493 at weights {'lgbm': 0.0, 'catboost_v1': 1.0, 'catboost_v2': 0.0, 'logreg': 0.0}
Best solo model:                  0.9493
Improvement (same-data, likely optimistic): +0.0000


In [13]:
nested_scores_solo_best = []
nested_scores_blend = []
nested_weights_4way = []
best_solo_key = max(solo_scores, key=solo_scores.get)

for fold_idx, (_, val_idx) in enumerate(tqdm(folds, desc='nested CV (4-way blend)')):
    other_idx = np.concatenate([folds[j][1] for j in range(N_FOLDS) if j != fold_idx])
    other_oof = {m: OOF_PROBAS[m][other_idx] for m in MEMBER_NAMES}
    _, w_fold = grid_search_blend(other_oof, y[other_idx], MEMBER_NAMES, grid4)

    val_oof = {m: OOF_PROBAS[m][val_idx] for m in MEMBER_NAMES}
    blend_pred = blend_predict(val_oof, w_fold, MEMBER_NAMES)
    solo_pred = OOF_PROBAS[best_solo_key][val_idx].argmax(axis=1)

    nested_scores_solo_best.append(balanced_accuracy_score(y[val_idx], solo_pred))
    nested_scores_blend.append(balanced_accuracy_score(y[val_idx], blend_pred))
    nested_weights_4way.append(w_fold)

print('Per-fold blend weights (fit on the other 4 folds):', [tuple(round(x, 2) for x in w) for w in nested_weights_4way])
print(f'Nested best-solo-model ({best_solo_key}) balanced accuracy: {np.mean(nested_scores_solo_best):.4f} (+/- {np.std(nested_scores_solo_best):.4f})')
print(f'Nested 4-way-blend balanced accuracy:                      {np.mean(nested_scores_blend):.4f} (+/- {np.std(nested_scores_blend):.4f})')
print(f'Honest improvement estimate: {np.mean(nested_scores_blend) - np.mean(nested_scores_solo_best):+.4f}')

nested CV (4-way blend):   0%|          | 0/5 [00:00<?, ?it/s]

blend grid (lgbm+catboost_v1+catboost_v2+logreg):   0%|          | 0/286 [00:00<?, ?it/s]

blend grid (lgbm+catboost_v1+catboost_v2+logreg):   0%|          | 0/286 [00:00<?, ?it/s]

blend grid (lgbm+catboost_v1+catboost_v2+logreg):   0%|          | 0/286 [00:00<?, ?it/s]

blend grid (lgbm+catboost_v1+catboost_v2+logreg):   0%|          | 0/286 [00:00<?, ?it/s]

blend grid (lgbm+catboost_v1+catboost_v2+logreg):   0%|          | 0/286 [00:00<?, ?it/s]

Per-fold blend weights (fit on the other 4 folds): [(0.0, 1.0, 0.0, 0.0), (0.3, 0.7, 0.0, 0.0), (0.0, 1.0, 0.0, 0.0), (0.0, 1.0, 0.0, 0.0), (0.2, 0.5, 0.3, 0.0)]
Nested best-solo-model (catboost_v1) balanced accuracy: 0.9493 (+/- 0.0011)
Nested 4-way-blend balanced accuracy:                      0.9492 (+/- 0.0011)
Honest improvement estimate: -0.0002


## Pairwise + triple blends (for comparison)

Worth knowing if a smaller subset does as well or better than the full 4-way blend —
e.g. if the two CatBoost variants are too correlated to add anything over one alone,
or if the logistic regression is too weak to help even with an optimal weight.

In [14]:
from itertools import combinations

subset_results = {}
for r in [2, 3]:
    for subset in combinations(MEMBER_NAMES, r):
        step = 0.02 if r == 2 else 0.05
        grid = simplex_grid(r, step=step)
        best_score, best_w = grid_search_blend(OOF_PROBAS, y, list(subset), grid)
        subset_results[subset] = (best_score, best_w)

for subset, (score, w) in sorted(subset_results.items(), key=lambda kv: -kv[1][0]):
    print(f'{score:.4f}  {"+".join(subset)}  weights={tuple(round(x,2) for x in w)}')

print(f'\n4-way blend (same-data): {best_score_4way:.4f}')
print(f'Best solo model:         {max(solo_scores.values()):.4f}')

blend grid (lgbm+catboost_v1):   0%|          | 0/51 [00:00<?, ?it/s]

blend grid (lgbm+catboost_v2):   0%|          | 0/51 [00:00<?, ?it/s]

blend grid (lgbm+logreg):   0%|          | 0/51 [00:00<?, ?it/s]

blend grid (catboost_v1+catboost_v2):   0%|          | 0/51 [00:00<?, ?it/s]

blend grid (catboost_v1+logreg):   0%|          | 0/51 [00:00<?, ?it/s]

...

blend grid (lgbm+catboost_v1+catboost_v2):   0%|          | 0/231 [00:00<?, ?it/s]

blend grid (lgbm+catboost_v1+logreg):   0%|          | 0/231 [00:00<?, ?it/s]

blend grid (lgbm+catboost_v2+logreg):   0%|          | 0/231 [00:00<?, ?it/s]

blend grid (catboost_v1+catboost_v2+logreg):   0%|          | 0/231 [00:00<?, ?it/s]

0.9493  lgbm+catboost_v1+catboost_v2  weights=(0.15, 0.45, 0.4)
0.9493  lgbm+catboost_v1  weights=(0.0, 1.0)
0.9493  catboost_v1+catboost_v2  weights=(1.0, 0.0)
0.9493  catboost_v1+logreg  weights=(1.0, 0.0)
0.9493  lgbm+catboost_v1+logreg  weights=(0.0, 1.0, 0.0)
0.9493  catboost_v1+catboost_v2+logreg  weights=(1.0, 0.0, 0.0)
0.9491  lgbm+catboost_v2  weights=(0.0, 1.0)
0.9491  catboost_v2+logreg  weights=(1.0, 0.0)
0.9491  lgbm+catboost_v2+logreg  weights=(0.0, 1.0, 0.0)
0.9442  lgbm+logreg  weights=(0.48, 0.52)

4-way blend (same-data): 0.9493
Best solo model:         0.9493


## Decision + candidate submission

Use the nested-validation estimate (not the full-OOF grid search) to decide whether
the ensemble is a real improvement over the best single model.

In [15]:
honest_improvement = np.mean(nested_scores_blend) - np.mean(nested_scores_solo_best)
THRESHOLD_FOR_REAL_IMPROVEMENT = 0.0005

if honest_improvement > THRESHOLD_FOR_REAL_IMPROVEMENT:
    print(f'REAL IMPROVEMENT: {honest_improvement:+.4f} (nested-validated). Using weights fit on full OOF: {dict(zip(MEMBER_NAMES, best_w_4way))}')
    test_pred = blend_predict(TEST_PROBAS, best_w_4way, MEMBER_NAMES)
    submission = pd.DataFrame({'id': test['id'], TARGET: label_encoder.inverse_transform(test_pred)})
    assert list(submission.columns) == list(sample_submission.columns)
    assert len(submission) == len(sample_submission)
    assert set(submission[TARGET].unique()) <= set(label_encoder.classes_)
    assert submission[TARGET].isnull().sum() == 0
    submission_path = DATA_DIR / 'submission.csv'
    submission.to_csv(submission_path, index=False)
    print(f'wrote {submission_path} ({len(submission)} rows) — NOT auto-submitted to Kaggle')
    display(submission[TARGET].value_counts(normalize=True))
else:
    print(f'NO REAL IMPROVEMENT: nested-validated delta {honest_improvement:+.4f} is within noise '
          f'(threshold {THRESHOLD_FOR_REAL_IMPROVEMENT}). Keeping the current best single model. '
          f'Not writing a new submission.csv.')

NO REAL IMPROVEMENT: nested-validated delta -0.0002 is within noise (threshold 0.0005). Keeping the current best single model. Not writing a new submission.csv.


## Summary

**Negative result — the 4-way ensemble does not beat CatBoost-V1 alone.**

All 3 reproductions PASS exact match: LightGBM 0.9389 (identical fold scores to
v0.1), CatBoost-V1 0.9493 (identical `best_iterations` to v0.3), CatBoost-V2 0.9491
(identical `best_iterations` to v0.3). The new logistic regression scored 0.8994
solo — notably weaker than the tree models, as expected for a linear model, though
it still shows the same class-weighting recall pattern (higher fit/unhealthy
recall, lower at-risk recall than an unweighted model would).

The 4-way blend weight search (simplex grid, 286 combinations) found its best
same-data score (0.9493) at weights that put 100% on CatBoost-V1 and 0% on
everything else — i.e. even the optimistic full-OOF fit found no benefit from
blending. Nested validation (fit weights on 4/5 folds, evaluate on the held-out
5th) confirmed this: honest improvement estimate **-0.0002**, within noise. Every
subset blend (all pairs and triples) that includes CatBoost-V1 caps out at exactly
its solo score; no combination beats it. `lgbm+logreg` (the only combination
without CatBoost-V1) tops out well below at 0.9442.

No new `submission.csv` was written, per the notebook's own decision threshold.

**Why**: this directly confirms the Rung 3 prediction that the competition data is
likely a noised synthesis of a near-deterministic depth-4 rule on
`sleep_duration`/`stress_level`/`physical_activity_level` (see
`docs/investigate/2026-07-03-kaggle-discussion-findings.md`, discussion 717222).
All four models here — including the architecturally distinct logistic regression
— already key on the same features and capture essentially all the recoverable
signal, leaving no complementary errors for an ensemble to correct. This is
stronger evidence for a synthesis-noise ceiling than Rung 3 alone, since it rules
out "wrong model family" as an explanation for the ~0.949-0.951 plateau.

**v0.3 (either variant) remains the best model at OOF ~0.949 / LB ~0.949.** Two
independent Rung 3/4 experiments now both point at the same ceiling — further
squeeze attempts should be weighed against this.